# Mạng MLP & Thuật toán Lan truyền ngược (Backpropagation) sử dụng PyTorch

Notebook này hướng dẫn bạn cách xây dựng mạng MLP và thực hiện lan truyền ngược bằng **PyTorch**.

### Điểm khác biệt lớn của PyTorch so với NumPy:
1. **Tensors**: Tương tự như NumPy arrays nhưng hỗ trợ tính toán trên GPU.
2. **Autograd (Automatic Differentiation)**: Tự động theo dõi các phép toán và tính toán đạo hàm (gradients) thông qua hàm `.backward()`. Bạn không cần viết tay công thức tính $\delta$ hay đạo hàm nữa!

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

# Thiết lập random seed để kết quả ổn định
torch.manual_seed(42)
np.random.seed(42)

print(f"Phiên bản PyTorch đang sử dụng: {torch.__version__}")

---
## PHẦN 1: TÍNH TOÁN THỦ CÔNG TỪNG BƯỚC VỚI PYTORCH AUTOGRAD

Ta sẽ khởi tạo một mạng có cấu trúc **3 → 4 → 4 → 2** như trên slide và in ra toàn bộ trọng số $W, b$ ban đầu.

In [ ]:
# Cấu trúc mạng (3 -> 4 -> 4 -> 2)
n_x = 3   # Input
n_h1 = 4  # Hidden 1
n_h2 = 4  # Hidden 2
n_y = 2   # Output

# Khởi tạo Trọng số (W) và Bias (b)
# Cần bật requires_grad=True để PyTorch theo dõi và tính đạo hàm tự động cho chúng ta
W1 = torch.randn(n_h1, n_x, dtype=torch.float32) * 0.1
W1.requires_grad = True
b1 = torch.zeros(n_h1, 1, dtype=torch.float32, requires_grad=True)

W2 = torch.randn(n_h2, n_h1, dtype=torch.float32) * 0.1
W2.requires_grad = True
b2 = torch.zeros(n_h2, 1, dtype=torch.float32, requires_grad=True)

W3 = torch.randn(n_y, n_h2, dtype=torch.float32) * 0.1
W3.requires_grad = True
b3 = torch.zeros(n_y, 1, dtype=torch.float32, requires_grad=True)

print("=== KHỞI TẠO TRỌNG SỐ (WEIGHTS & BIASES) ===")
print("\nW1:\n", W1.data)
print("b1:\n", b1.data)
print("\nW2:\n", W2.data)
print("b2:\n", b2.data)
print("\nW3:\n", W3.data)
print("b3:\n", b3.data)

### Bước 1: Đầu vào mạng ($a^0 = x$)

In [ ]:
# Khởi tạo vector input x (không cần tính đạo hàm cho input nên requires_grad=False)
x = torch.randn(n_x, 1, dtype=torch.float32)
a0 = x
print("Đầu vào a0:\n", a0)

### Bước 2: Lan truyền tiến từng lớp (Forward Propagation)
Sử dụng toán tử `@` trong PyTorch thay thế cho `np.dot` của NumPy.

In [ ]:
print("=== LAN TRUYỀN TIẾN (FORWARD PASS) ===\n")

# Lớp ẩn 1
z1 = W1 @ a0 + b1
a1 = torch.relu(z1)
print("z1:\n", z1.data)
print("a1 (ReLU):\n", a1.data)

# Lớp ẩn 2
z2 = W2 @ a1 + b2
a2 = torch.relu(z2)
print("\nz2:\n", z2.data)
print("a2 (ReLU):\n", a2.data)

# Lớp đầu ra
z3 = W3 @ a2 + b3
a3 = torch.sigmoid(z3)
y_hat = a3
print("\nz3:\n", z3.data)
print("y_hat (Sigmoid):\n", y_hat.data)

### Bước 3: Hàm mất mát (Loss) và Lan truyền ngược tự động (Backward)
Ta chỉ cần định nghĩa hàm Loss, sau đó gọi `.backward()` để tự động tính gradient cho toàn bộ mạng.

In [ ]:
# Nhãn thực tế y
y = torch.tensor([[0.5], [0.1]], dtype=torch.float32)

# Tính MSE Loss: 1/2 * sum((y_hat - y)^2)
loss = 0.5 * torch.sum((y_hat - y) ** 2)
print(f"Giá trị Loss: {loss.item():.6f}\n")

# Lan truyền ngược để tính toán gradients tự động
loss.backward()

print("=== GRADIENTS ĐÃ TỰ ĐỘNG TÍNH (dLoss/dW & dLoss/db) ===")
print("\ndW3 (W3.grad):\n", W3.grad)
print("db3 (b3.grad):\n", b3.grad)

print("\ndW2 (W2.grad):\n", W2.grad)
print("db2 (b2.grad):\n", b2.grad)

print("\ndW1 (W1.grad):\n", W1.grad)
print("db1 (b1.grad):\n", b1.grad)

---
## PHẦN 2: ĐỊNH NGHĨA MẠNG MLP SỬ DỤNG PYTORCH API NATIVE

Trong thực tế, ta thường thừa kế class `nn.Module` của PyTorch để tạo mạng nơ-ron một cách chuyên nghiệp.

In [ ]:
class PyTorchMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim1, hidden_dim2, output_dim):
        super(PyTorchMLP, self).__init__()
        
        # Định nghĩa các lớp liên kết đầy đủ (Dense / Linear)
        self.fc1 = nn.Linear(input_dim, hidden_dim1)
        self.fc2 = nn.Linear(hidden_dim1, hidden_dim2)
        self.fc3 = nn.Linear(hidden_dim2, output_dim)
        
        # Các hàm kích hoạt
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x):
        # Lan truyền tiến qua mạng
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)
        out = self.relu(out)
        out = self.fc3(out)
        out = self.sigmoid(out)
        return out

# Tạo một instance của mô hình
model = PyTorchMLP(input_dim=3, hidden_dim1=4, hidden_dim2=4, output_dim=2)
print(model)

### Huấn luyện và trực quan hóa mô hình trên tập dữ liệu Moons

In [ ]:
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Chuẩn bị dữ liệu bằng scikit-learn
X, y = make_moons(n_samples=400, noise=0.2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Chuyển đổi dữ liệu sang dạng PyTorch Tensors
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1) # Chuyển thành (N, 1)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

# 2. Khởi tạo mô hình (Mạng phân loại nhị phân: 2 -> 8 -> 4 -> 1)
clf_model = PyTorchMLP(input_dim=2, hidden_dim1=8, hidden_dim2=4, output_dim=1)

# Chọn hàm tối ưu (Optimizer) và Hàm mất mát (Loss Function)
criterion = nn.MSELoss() # Có thể dùng nn.BCELoss() trong thực tế
optimizer = optim.SGD(clf_model.parameters(), lr=0.2)

# Vòng lặp huấn luyện
epochs = 2000
loss_history = []
accuracy_history = []

for epoch in range(epochs):
    # Đặt mô hình ở chế độ huấn luyện
    clf_model.train()
    
    # Xóa sạch các gradient cũ
    optimizer.zero_grad()
    
    # Tính toán Forward Pass
    predictions = clf_model(X_train_t)
    
    # Tính toán Loss
    loss = criterion(predictions, y_train_t)
    loss_history.append(loss.item())
    
    # Tính độ chính xác
    preds_bin = (predictions >= 0.5).float()
    acc = (preds_bin == y_train_t).float().mean().item()
    accuracy_history.append(acc)
    
    # Tính toán lan truyền ngược
    loss.backward()
    
    # Cập nhật trọng số
    optimizer.step()
    
    if (epoch + 1) % 200 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:4d}/{epochs} - Loss: {loss.item():.6f} - Acc: {acc:.2%}")

In [ ]:
# 3. Đánh giá trên tập kiểm thử (Test set)
clf_model.eval()
with torch.no_grad():
    test_outputs = clf_model(X_test_t)
    test_preds = (test_outputs >= 0.5).float()
    test_acc = (test_preds == y_test_t).float().mean().item()
    print(f"\nTest Accuracy: {test_acc:.2%}")

# 4. Vẽ đồ thị kết quả huấn luyện
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Đồ thị Loss & Accuracy
axes[0].plot(loss_history, color='tab:red', label='Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss (MSE)', color='tab:red')
axes[0].tick_params(axis='y', labelcolor='tab:red')

ax2 = axes[0].twinx()
ax2.plot(accuracy_history, color='tab:blue', label='Accuracy')
ax2.set_ylabel('Accuracy', color='tab:blue')
ax2.tick_params(axis='y', labelcolor='tab:blue')
axes[0].set_title("Lịch sử Huấn luyện (PyTorch)")
axes[0].grid(True, alpha=0.3)

# Đồ thị Decision Boundary
x_min, x_max = X_test[:, 0].min() - 0.5, X_test[:, 0].max() + 0.5
y_min, y_max = X_test[:, 1].min() - 0.5, X_test[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))
grid = np.c_[xx.ravel(), yy.ravel()]
grid_t = torch.tensor(grid, dtype=torch.float32)

with torch.no_grad():
    probs = clf_model(grid_t).reshape(xx.shape).numpy()

axes[1].contourf(xx, yy, probs, levels=50, cmap='RdYlBu', alpha=0.6)
scatter = axes[1].scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap='RdYlBu', edgecolors='black', s=40)
axes[1].set_title(f"Ranh giới Quyết định (Test Acc: {test_acc:.2%})")
plt.colorbar(scatter, ax=axes[1])
axes[1].grid(True, alpha=0.3)

plt.show()